In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.Message import UserMessage

from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill

In [2]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)
# agent.with_skill(CalculatorSkill())
print(llm.model)

2026-04-20 15:20:07,734 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b
2026-04-20 15:20:07,874 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: openai


qwen3.5-9b


In [ ]:
# llm.invoke_raw([UserMessage("你是?")])
agent.invoke("请仔细思考,你是?")

In [ ]:
agent.get_history()

In [ ]:
await agent.astream_invoke("你是?")

In [ ]:
#自定义skill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""


In [ ]:
agent.with_skill(TranslateSkill())


In [ ]:
from core import enable_logging
enable_logging()
agent.clear_history()
# agent._build_start_messages(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22" )

In [ ]:
await agent.astream_invoke("我们刚才说了什么")

In [ ]:
agent.get_history()

In [ ]:
message=agent._build_start_messages("111")
agent.llm._convert_messages(message)

In [ ]:
print(agent.get_enhanced_prompt())

In [ ]:
agent.get_trace_history()

In [ ]:
agent.save_session("test_00001")

In [ ]:
agent2=BasicAgent.load_session("test_00001",llm=agent.llm)

In [ ]:
from skill import SkillManager


agent_resume:BasicAgent=BasicAgent.load_session("test_00001",llm=agent.llm,tool_registry=agent.tool_registry,skill_manager=agent.skill_manager)

In [ ]:
await agent_resume.astream_invoke("我们刚才聊了什么")

In [ ]:
agent_resume.get_trace_history()

In [ ]:
manager=agent.skill_manager
prompt=manager.build_skills_prompt()
print(prompt)

In [ ]:
from skill.registry import SkillRegistry
skill_manage=SkillRegistry()
skill_manage.discover_from_directory("./real_skills/")



In [ ]:
print(skill_manage.list_available())


In [ ]:
crypto_skill=skill_manage.create('crypto_skill')
agent.with_skill(crypto_skill)
print(agent.get_enhanced_prompt())

In [ ]:
agent.invoke("i am a boy from china的 SHA-256 哈希值是什么")

In [3]:
from memory.V2.WorkingMemory import WorkingMemory
from memory import MemoryConfig,MemoryManage
from memory.V2.Embedding.HuggingfaceEmbeddingModel import HuggingfaceEmbeddingModel
config = MemoryConfig(max_capacity=20)
working_memory = WorkingMemory(config)
mm = MemoryManage(
            config=config,
            user_id="test_integration_user",
            enable_working=True,
            working_memory=working_memory,
            enable_episodic=False,
            enable_semantic=False,
            enable_perceptual=False,
        ) 

/home/wxd/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-20 15:20:18,414 | INFO | MemoryManage init success
2026-04-20 15:20:18,415 | INFO | MemoryManage init success, memory types: dict_keys(['working'])


In [ ]:
agent.with_memory(mm)
print(agent.get_enhanced_prompt())

SkillManager(registered=[], active=[])

In [ ]:
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

# 1. 把所有 Skill 注册到全局 Registry（启动时一次性完成）
registry = SkillRegistry.instance()
registry.discover_from_directory("./real_skills/")
# 也可以从目录批量发现
# registry.discover_from_directory("./skills/")

# 2. 创建 Agent（不预加载任何 Skill）
agent1 = BasicAgent(name="assistant", llm=llm, verbose_thinking=True)
agent1.with_skill(MetaSkill(registry,manager=agent1.skill_manager))
print(agent1.get_enhanced_prompt())

In [ ]:
await agent1.astream_invoke("i am a boy from china的 SHA-256 哈希值是什么")

In [ ]:
agent1.get_raw_history()

In [ ]:
from context import ContextManager,ContextBuilder,LLMHistoryCompactor
from skill.registry import SkillRegistry
from core import enable_logging
enable_logging()
llm2= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

skill_manage=SkillRegistry()
skill_manage.discover_from_directory("./real_skills/")
crypto_skill=skill_manage.create('crypto_skill')

agent_context = BasicAgent(name="assistant", llm=llm2,reasoning={"effort":"high"} ,verbose_thinking=True)    
agent_context.with_skill(crypto_skill)
builder=ContextManager(max_tokens=3000)
builder.set_history_compactor(LLMHistoryCompactor(llm2,recent_turns=1))
agent_context.with_context(builder)


In [ ]:
agent_context.get_context_usage()

In [ ]:
agent_context.invoke("i am a boy from acc SHA-256 哈希值是什么")


In [ ]:
agent_context.get_context_usage()

In [ ]:
len(agent_context.get_canonical_history())

In [ ]:
cm=LLMHistoryCompactor(llm2,recent_turns=0)
re=cm.compact(agent_context.get_canonical_history(),max_tokens=300)